In [1]:
from stable_baselines3 import PPO
import torch
import numpy as np

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:
class OnnxableActionPolicy(torch.nn.Module):
    def __init__(self, extractor, action_net, value_net):
        super(OnnxableActionPolicy, self).__init__()
        self.extractor = extractor
        self.action_net = action_net
        self.value_net = value_net
        normalize_linear1 = torch.nn.Linear(1, 4)
        # 100* ((max(0,x) - max(0,-x)) - max(0,x-1) + max(0,-x-1))
        normalize_linear1.weight.data = torch.Tensor([[1],[-1],[1],[-1]])
        normalize_linear1.bias.data=torch.Tensor([0,0,-1,-1])
        #print(normalize_linear1.weight)
        #print(normalize_linear1.bias)
        A = 100-1e-6
        normalize_linear2 = torch.nn.Linear(3,1)
        normalize_linear2.weight.data = torch.Tensor([[A,-A,-A,A]])
        normalize_linear2.bias.data=torch.Tensor([0])
        self.normalizer = torch.nn.Sequential(
            normalize_linear1,
            torch.nn.ReLU(),
            normalize_linear2)

    def forward(self, observation):
        # NOTE: You may have to process (normalize) observation in the correct
        #       way before using this. See `common.preprocessing.preprocess_obs`
        action_hidden, value_hidden = self.extractor(observation)
        action = self.action_net(action_hidden)
        return self.normalizer(action) #, self.value_net(value_hidden)

In [38]:
# Example: model = PPO("MlpPolicy", "Pendulum-v0")
model = PPO.load("ppo_acc_small_200000_steps.zip")
model.policy.to("cpu")
onnxable_model = OnnxableActionPolicy(model.policy.mlp_extractor, model.policy.action_net, model.policy.value_net)

In [6]:
onnxable_model.normalizer(torch.Tensor([2]))

tensor([100.], grad_fn=<ViewBackward0>)

In [42]:
import onnx

dummy_input = torch.randn(1, 2)
torch.onnx.export(
    onnxable_model,
    dummy_input,
    "ppo_acc_small_200000_steps.onnx",
    opset_version=9,
    dynamo=False,  # use legacy exporter so requested opset is honored
    input_names=["observation"],
    output_names=["out1"],
)

tmp_model = onnx.load("ppo_acc_small_200000_steps.onnx")
print("exported opset:", [(o.domain, o.version) for o in tmp_model.opset_import])

exported opset: [('', 9)]


C:\Users\ghasob\AppData\Local\Temp\ipykernel_269688\4216648265.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [43]:
##### Load and test with onnx

import onnx
import onnxruntime as ort
import numpy as np

In [44]:
onnx_model = onnx.load("ppo_acc_small_200000_steps.onnx")
onnx.checker.check_model(onnx_model)

In [45]:
onnx_model = onnx.load("ppo_acc_small_200000_steps.onnx")
onnx.checker.check_model(onnx_model)

observation = np.zeros((1, 2)).astype(np.float32)
ort_sess = ort.InferenceSession("ppo_acc_small_200000_steps.onnx")

In [46]:
print(ort_sess.run(None, {'observation': [[50, -99.0]]}))

[array([[100.00001]], dtype=float32)]


In [47]:
import gymnasium as gym
import acc

In [48]:
env = gym.make("acc-variant-v1")

c:\Users\ghasob\AppData\Local\Programs\Python\Python313\Lib\site-packages\gymnasium\envs\registration.py:512: DeprecationWarning: WARN: The environment acc-variant-v1 is out of date. You should consider upgrading to version `v2`.
  logger.deprecation(


In [49]:
obs = env.reset()
obs = [50,-99.0]
env.unwrapped.state = obs
for i in range(0,300):
    action = ort_sess.run(None, {'observation': [[obs[0],obs[1]]]})[0][0]
    #action = np.clip(action,-100.,100.)
    obs, rewards, dones, truncated, info = env.step(action)
    #env.render()
    if dones or truncated:
        print("DONE")
        print(obs)
        env.reset()

In [16]:
del env

# Rename output nodes to not purely numeric names!

In [50]:
onnx_model = onnx.load('ppo_acc_small_200000_steps.onnx')

In [51]:
onnx_model.graph.output[0].name = "out1"

In [52]:
onnx_model.graph.node[len(onnx_model.graph.node)-1].output[0]="out1"

In [53]:
onnx_model.graph.input[0].name

'observation'

In [54]:
onnx.save(onnx_model, 'ppo_acc_small_200000_steps.onnx')